In [2]:
# Cell 1: Read the silver_sales Delta Table
df_silver = spark.read.format('Delta').load('Tables/dbo/silver_sales')

print(f'Row count: {df_silver.count()}')
print(f'Columns: {df_silver.columns}')
display(df_silver.limit(5))

StatementMeta(, e3ee14d8-381a-45d1-96c2-e1cfd9ffccd0, 4, Finished, Available, Finished, False)

Row count: 1268
Columns: ['order_id', 'order_date', 'customer_name', 'region', 'product_category', 'revenue', 'Quantity', 'status', 'revenue_usd']


SynapseWidget(Synapse.DataFrame, 226a3aee-c7d0-4acf-85d6-32bf6c3eb1cc)

In [3]:
# Cell 2: Extract year and month from order_date
from pyspark.sql.functions import col, round, date_format, sum, avg, count

# Add year_month column for grouping
df_dated = df_silver.withColumn(
    'year_month',
    date_format(col('order_date'), 'yyyy-MM')
)

display(df_dated.select('order_id', 'order_date', 'year_month').limit(5))


StatementMeta(, e3ee14d8-381a-45d1-96c2-e1cfd9ffccd0, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f9caab47-3094-4e72-9a66-e4cc973ab28e)

In [4]:
# Cell 3: Aggregate to Gold layer
df_gold = df_dated.groupBy('region', 'year_month') \
    .agg(
        round(sum('revenue'), 2).alias('total_revenue'),
        count('order_id').alias('total_orders'),
        round(avg('revenue'), 2).alias('avg_order_value')
    ) \
    .orderBy('year_month', 'region')

print(f'Gold Table Row Count: {df_gold.count()}')
display(df_gold)

StatementMeta(, e3ee14d8-381a-45d1-96c2-e1cfd9ffccd0, 6, Finished, Available, Finished, False)

Gold Table Row Count: 96


SynapseWidget(Synapse.DataFrame, e750510d-3c9c-4818-9aa6-baa53824aeb5)

In [5]:
# Cell 4: Write the Gold Layer to a Delta Table
df_gold.write \
    .format('delta') \
    .mode('overwrite') \
    .saveAsTable('gold_sales_summary')
    

StatementMeta(, e3ee14d8-381a-45d1-96c2-e1cfd9ffccd0, 7, Finished, Available, Finished, False)